<a href="https://colab.research.google.com/github/yorique/colab/blob/main/InspecaoB24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import ee
ee.Authenticate()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path('/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2')
print('PROJECT_ROOT_EXISTS', PROJECT_ROOT.is_dir())
for p in sorted(PROJECT_ROOT.iterdir()):
    print(('DIR ' if p.is_dir() else 'FILE'), p.name)

PROJECT_ROOT_EXISTS True
DIR  Artefatos Operacionais
DIR  Auditorias
DIR  Checkpoints
DIR  Configurações
DIR  Limites
DIR  Modelos
DIR  Relatórios
DIR  Resultados Finais
DIR  Staging
DIR  Tabelas Consolidadas
DIR  Validação Externa


In [6]:
import os

keywords = ('b24', 'bloco 24', 'conservativa', 'conservacao', 'conservação')
for current, dirs, files in os.walk(PROJECT_ROOT):
    depth = len(Path(current).relative_to(PROJECT_ROOT).parts)
    if depth >= 3:
        dirs[:] = []
    selected = [name for name in dirs + files if any(k in name.lower() for k in keywords)]
    if selected:
        print('AT', Path(current).relative_to(PROJECT_ROOT))
        for name in selected:
            p = Path(current) / name
            print(' ', 'DIR' if p.is_dir() else 'FILE', name, p.stat().st_size if p.is_file() else '')

AT Configurações
  FILE Configuracao_Bloco12_V2_Correcao_Conservativa_Caete.json 3963
  FILE Configuracao_Bloco24_V2_Correcao_Conservativa.json 6284
AT Relatórios
  FILE Relatorio_Bloco12_V2_Correcao_Conservativa_Caete.txt 2868
  FILE Relatorio_Bloco24_V2_Correcao_Conservativa.txt 11462
AT Auditorias
  DIR Bloco 24 
  FILE Auditoria_Pos_Escrita_Conservacao_Bloco12_V2.csv 33603
  FILE Testes_Bloco12_V2_Correcao_Conservativa_Caete.csv 1359
  FILE Diagnostico_Correcao_Conservativa_Bloco12_V2.csv 352451
  FILE Testes_Bloco24_V2_Correcao_Conservativa.csv 2157
AT Auditorias/Bloco 24
  FILE Diagnostico_Correcao_Conservativa_10_Bacias_Bloco24_V2.csv 1912995
  FILE Resumo_Conservacao_10_Bacias_Bloco24_V2.csv 1843
  FILE Resumo_Fatores_Conservacao_10_Bacias_Bloco24_V2.csv 957
  FILE Testes_Bloco24_V2_Correcao_Conservativa.csv 2157
  FILE Configuracao_Bloco24_V2_Correcao_Conservativa.json 6284
  FILE Relatorio_Bloco24_V2_Correcao_Conservativa.txt 11462
AT Auditorias/Bloco 12
  FILE Auditoria_P

In [7]:
B24_STAGE = PROJECT_ROOT / 'Staging' / 'Bloco 24 V2 - Correcao Conservativa Suave 250m Multibacias'
for p in sorted(B24_STAGE.iterdir()):
    print(('DIR ' if p.is_dir() else 'FILE'), p.name, p.stat().st_size if p.is_file() else '')

print('\nCODE_CANDIDATES')
for base in [PROJECT_ROOT / 'Artefatos Operacionais', PROJECT_ROOT / 'Checkpoints', PROJECT_ROOT / 'Staging']:
    if base.exists():
        for current, dirs, files in os.walk(base):
            depth = len(Path(current).relative_to(base).parts)
            if depth >= 3: dirs[:] = []
            for name in files:
                if Path(name).suffix.lower() in {'.py', '.ipynb', '.txt', '.r'} and ('24' in name.lower() or 'conserv' in name.lower()):
                    p = Path(current)/name
                    print(p, p.stat().st_size)

DIR  _Caches_Overlap_RBF 
DIR  _Checkpoints 
DIR  apeu 
DIR  caripi 
DIR  guama 
DIR  maracana 
DIR  marapanim 
DIR  morcego 
DIR  piria 
DIR  quatipuru 
DIR  taperacu 
DIR  urumajo 

CODE_CANDIDATES


In [8]:
import json, hashlib, pandas as pd

def digest(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

paths = {
    'config': PROJECT_ROOT / 'Configurações' / 'Configuracao_Bloco24_V2_Correcao_Conservativa.json',
    'report': PROJECT_ROOT / 'Relatórios' / 'Relatorio_Bloco24_V2_Correcao_Conservativa.txt',
    'tests': PROJECT_ROOT / 'Auditorias' / 'Bloco 24' / 'Testes_Bloco24_V2_Correcao_Conservativa.csv',
    'summary': PROJECT_ROOT / 'Auditorias' / 'Bloco 24' / 'Resumo_Conservacao_10_Bacias_Bloco24_V2.csv',
}
for label, p in paths.items():
    print('ARTIFACT', label, p.name, 'bytes=', p.stat().st_size, 'sha256=', digest(p), 'mtime=', datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec='seconds'))

print('\nCONFIG')
print(json.dumps(json.loads(paths['config'].read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:12000])
print('\nREPORT')
print(paths['report'].read_text(encoding='utf-8')[:15000])
print('\nTESTS')
print(pd.read_csv(paths['tests']).to_string(index=False))
print('\nSUMMARY')
print(pd.read_csv(paths['summary']).to_string(index=False))

ARTIFACT config Configuracao_Bloco24_V2_Correcao_Conservativa.json bytes= 6284 sha256= cf1c0e027e670f239926e091e30995515896ab2b75ce9a4e6a2db3216a07879f mtime= 2026-08-21T13:11:55
ARTIFACT report Relatorio_Bloco24_V2_Correcao_Conservativa.txt bytes= 11462 sha256= f880be36ca9cd0a84eb768f7ca82d360be4d0d3b1a598fba486a4dd4b68c9915 mtime= 2026-08-21T13:11:55
ARTIFACT tests Testes_Bloco24_V2_Correcao_Conservativa.csv bytes= 2157 sha256= 50b6330185ef4bb5f77bcaceac62687235c15a9e89efcda83a7c4e5282dfd71b mtime= 2026-08-31T12:31:47
ARTIFACT summary Resumo_Conservacao_10_Bacias_Bloco24_V2.csv bytes= 1843 sha256= 030e722e00ddbe2fdfa26ced271959c20eb54632d21a7747078ee6c117b8f2d3 mtime= 2026-08-31T12:31:47

CONFIG
{
  "bloco": "BLOCO_24_V2",
  "fase": "FASE_B_CORRECAO_CONSERVATIVA_GAUSSIANA",
  "equivalencia_amcdm": "D11_V5",
  "versao": "BLOCO_24_V2_v1.0.1_CORRECAO_1_PREFLIGHT_FINE_SUPPORT",
  "versao_execucao": "BLOCO_24_V2_v1.0.7_CORRECAO_7_RELAXAMENTO_ESTRUTURAL_IDENTIFICAVEL",
  "identidade_produt

In [9]:
audit_dir = PROJECT_ROOT / 'Auditorias' / 'Bloco 24'
checks = {
    'config_copia_auditoria': audit_dir / 'Configuracao_Bloco24_V2_Correcao_Conservativa.json',
    'relatorio_copia_auditoria': audit_dir / 'Relatorio_Bloco24_V2_Correcao_Conservativa.txt',
    'manifesto_referenciado': PROJECT_ROOT / 'Auditorias' / 'Manifesto_Bloco24_V2.csv',
    'inventario_referenciado': PROJECT_ROOT / 'Auditorias' / 'Inventario_Rasters_Corrigidos_Suaves_Bloco24_V2.csv',
    'preflight_referenciado': PROJECT_ROOT / 'Tabelas Consolidadas' / 'Preflight_Representacao_Fina_10_Bacias_Bloco24_V2.csv',
}
for k,p in checks.items():
    print(k, 'EXISTS=',p.is_file(), 'sha256='+(digest(p) if p.is_file() else 'NA'))

for k, p in [('config', paths['config']), ('relatorio', paths['report'])]:
    q = checks[f'{k}_copia_auditoria']
    print('COPY_IDENTITY', k, q.is_file() and digest(p)==digest(q))

checkpoint_files = sorted((B24_STAGE / '_Checkpoints').glob('Checkpoint_B24_*.json'))
print('CHECKPOINTS', len(checkpoint_files))
for p in checkpoint_files:
    item = json.loads(p.read_text(encoding='utf-8'))
    selected = {k:item.get(k) for k in item if any(x in k.lower() for x in ('vers', 'data', 'time', 'hash', 'bacia', 'status', 'relax', 'restr'))}
    print(p.name, json.dumps(selected, ensure_ascii=False)[:1500])

config_copia_auditoria EXISTS= True sha256=cf1c0e027e670f239926e091e30995515896ab2b75ce9a4e6a2db3216a07879f
relatorio_copia_auditoria EXISTS= True sha256=f880be36ca9cd0a84eb768f7ca82d360be4d0d3b1a598fba486a4dd4b68c9915
manifesto_referenciado EXISTS= True sha256=fbbaf85753efebd44ff68f946f38b9652cbbcc9b3b8fd09ae294d8ba25a57737
inventario_referenciado EXISTS= True sha256=bdbf8611b8c39547899ad7fd37a314c90abb9981b8a1c34c954402176c8d727b
preflight_referenciado EXISTS= True sha256=95747355a1bb481808de808ed8ed8ea582e81477a90835e5d53ba7366228261f
COPY_IDENTITY config True
COPY_IDENTITY relatorio True
CHECKPOINTS 10
Checkpoint_B24_apeu.json {"versao": "BLOCO_24_V2_v1.0.1_CORRECAO_1_PREFLIGHT_FINE_SUPPORT", "data_utc": "2026-08-21T00:21:16+00:00", "bacia": "apeu"}
Checkpoint_B24_caripi.json {"versao": "BLOCO_24_V2_v1.0.1_CORRECAO_1_PREFLIGHT_FINE_SUPPORT", "data_utc": "2026-08-21T00:22:41+00:00", "bacia": "caripi"}
Checkpoint_B24_guama.json {"versao": "BLOCO_24_V2_v1.0.1_CORRECAO_1_PREFLIGHT_FINE

In [10]:
manifest = checks['manifesto_referenciado']
print(manifest.read_text(encoding='utf-8')[:20000])
print('\nINVENTORY_HEAD')
print(checks['inventario_referenciado'].read_text(encoding='utf-8')[:5000])
print('\nPREFLIGHT')
print(checks['preflight_referenciado'].read_text(encoding='utf-8')[:5000])

﻿Arquivo;Caminho;Existe;Tamanho_bytes;SHA256
Auditoria_Bloco23_no_Bloco24_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Auditoria_Bloco23_no_Bloco24_V2.csv;True;685;97676e59733abf268598f1dd83b3ec6b8a00f2300fc402e661de2d26f9c2638d
Auditoria_Rasters_Raw_B23_no_Bloco24_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Auditoria_Rasters_Raw_B23_no_Bloco24_V2.csv;True;58616;a0608a9f7e2aa756461113365a52845ac9ca0da0238ed21e5d46d103a99b9790
Auditoria_Entradas_CHIRPS_B15_no_Bloco24_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Auditoria_Entradas_CHIRPS_B15_no_Bloco24_V2.csv;True;486;641a9c8fd051523c3594719b71d6ab39492937a1225ded5906118fc111c366ef
Preflight_Representacao_Fina_10_Bacias_Bloco24_V2.csv;/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Tabelas Consolidadas/Preflight_Representacao_Fina_10_Bacias_Bloco24_V2.csv;True;852;95747355a1bb481808de808ed8ed8ea582e81477a90835e5

In [2]:
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive')
patterns = ('b24', 'bloco 24', 'correcao conservativa', 'correção conservativa')
hits = []
for path in DRIVE_ROOT.rglob('*'):
    if path.is_file() and any(p in path.name.lower() for p in patterns):
        stat = path.stat()
        hits.append({
            'path': str(path),
            'bytes': stat.st_size,
            'modified': datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds')
        })
print('B24_CANDIDATES', len(hits))
for item in sorted(hits, key=lambda x: x['path']):
    print(item)

B24_CANDIDATES 51
{'path': '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Staging/Bloco 24 V2 - Correcao Conservativa Suave 250m Multibacias/_Caches_Overlap_RBF/Overlap_apeu_B24_C1.json', 'bytes': 2938, 'modified': '2026-08-21T00:19:01'}
{'path': '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Staging/Bloco 24 V2 - Correcao Conservativa Suave 250m Multibacias/_Caches_Overlap_RBF/Overlap_apeu_B24_C1.npz', 'bytes': 73429, 'modified': '2026-08-21T00:19:00'}
{'path': '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Staging/Bloco 24 V2 - Correcao Conservativa Suave 250m Multibacias/_Caches_Overlap_RBF/Overlap_caripi_B24_C1.json', 'bytes': 2808, 'modified': '2026-08-21T00:19:05'}
{'path': '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Staging/Bloco 24 V2 - Correcao Conservativa Suave 250m Multibacias/_Caches_Overlap_RBF/Overlap_caripi_B24_C1.npz', 'bytes': 68420, 'modified': '2026-08-21T00:19:05'}
{'path': '/content/drive/M